# Proposed Model 4 — SAM2 + Grounding DINO + SigLIP 2\n\n**Pipeline:** Upload RGB video → click one target → SAM2 tracking → global lost-target detection → VLM identity matching → two-search temporal confirmation → SAM2 relock, or stay SEARCHING if uncertain.\n\nOptional target type field supports labels like 'car' or 'person'; leave empty for SigLIP 2 auto-category. Select a **GPU** runtime. The first model download may take considerable disk space and RAM.\n\n**No ground-truth accuracy is claimed**; identity ambiguity cannot always be resolved.

In [ ]:
import os,sys,subprocess,urllib.request
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. Runtime > Change runtime type > GPU.")

REPO=Path("/content/Target_Locker")
BRANCH="feature/proposed-model-4-vlm-memory"
MODEL_DIR=REPO/"proposed_model_4"
SAM_CKPT=Path("/content/sam2.1_hiera_tiny.pt")

if not REPO.exists():
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/CptImtiaz/Target_Locker.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","--depth","1","origin",BRANCH],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","-B",BRANCH,"FETCH_HEAD"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",
                str(MODEL_DIR/"requirements.txt")],check=True)

if not SAM_CKPT.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        SAM_CKPT)
for name in ["app_interface.py","model4_tracker.py","reacquisition.py",
             "vlm_identity.py","confidence_policy.py","video_io.py"]:
    assert (MODEL_DIR/name).is_file(),name
sys.path.insert(0,str(MODEL_DIR))
print("Proposed Model 4 ready:",MODEL_DIR)
print("SAM2 checkpoint:",SAM_CKPT)
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
import sys,subprocess
from pathlib import Path
directory=Path("/content/Target_Locker/proposed_model_4")
subprocess.run([sys.executable,"-m","unittest","discover","-s",
                str(directory/"tests"),"-p","test_*.py","-v"],check=True)


In [ ]:
import os,sys,runpy
from pathlib import Path
directory=Path("/content/Target_Locker/proposed_model_4")
os.chdir(directory)
if str(directory) not in sys.path:sys.path.insert(0,str(directory))
for module in ["video_io","vlm_identity","reacquisition","confidence_policy","model4_tracker"]:
    sys.modules.pop(module,None)
runpy.run_path(str(directory/"app_interface.py"),run_name="__main__")
